# 🧱 Lab: Backend Swap Comparison

**Module 6: The Replaceable Model Interface** | **Type: Wall Lab**

---

## Learning Objectives

By the end of this lab, you will be able to:

1. **Run** identical input rows through two backends using the same interface
2. **Record** disagreements and timings in a comparison file
3. **Swap** backends by configuration — mock vs hosted-or-fixture on the required path
4. **Tag** `v3` once the recorded diff exists

## Concepts Covered

| Concept | Gloss | From |
|---------|-------|------|
| Interface Seam | Small typed contract; adapter returns your dataclass | mini-define-ai-protocol |
| Dependency Injection | Pass the backend in; pipeline code stays unchanged | mini-define-ai-protocol |
| HTTP API Request | Base URL, env-var key, timeout; fixtures when no key | mini-hosted-backend |
| Open vs Closed Models | Run it yourself vs reach it through a provider | This lab |
| Baseline Comparison | Judge output against a simple reference | mini-run-first-triage |

## Prerequisites

- **mini-define-ai-protocol**: Protocol + injected mock
- **mini-typed-prompt-parser**: Versioned prompt and defensive parse
- **mini-hosted-backend**: Hosted adapter with fixture fallback

No paid API key and no Ollama install are required.

## Setup

Import deps and define the shared seam/backends used in this lab.

In [3]:
import os
import json
import time
from dataclasses import dataclass, field, asdict
from typing import Protocol, runtime_checkable, Optional

import requests
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv()


def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

## Step 1 — Shared Types and Backends

We reuse `AIRequest`/`AIResponse`/`AIClient` and define two adapters: `MockAIClient` and `HostedAIClient` (with fixture fallback).

> 🧩 **Open vs Closed Models.** Open models you run yourself; closed ones live behind a provider's API. Watch which adapter here is the closed one.

In [4]:
# ── Shared types ──

@dataclass
class AIRequest:
    prompt: str
    system: str = "You are a helpful assistant."
    temperature: float = 0.0
    max_tokens: int = 256


@dataclass
class AIResponse:
    text: str
    model: str = "unknown"
    usage: dict = field(default_factory=dict)


@runtime_checkable
class AIClient(Protocol):
    def complete(self, request: AIRequest) -> AIResponse: ...


# ── Hosted backend (OpenAI) ──

class HostedAIClient:
    def __init__(self, api_key: str, base_url: str = "https://api.openai.com/v1",
                 model: str = "gpt-4o-mini", timeout: int = 30):
        self.api_key = api_key or ""
        self.base_url = base_url.rstrip("/")
        self.model = model
        self.timeout = timeout

    def complete(self, request: AIRequest) -> AIResponse:
        url = f"{self.base_url}/chat/completions"
        headers = {"Authorization": f"Bearer {self.api_key}",
                   "Content-Type": "application/json"}
        payload = {
            "model": self.model,
            "messages": [
                {"role": "system", "content": request.system},
                {"role": "user", "content": request.prompt},
            ],
            "temperature": request.temperature,
            "max_tokens": request.max_tokens,
        }
        if not self.api_key or self.api_key.startswith("sk-fixture"):
            label = "billing" if "charg" in request.prompt.lower() else "other"
            return AIResponse(text=label, model=f"{self.model}-fixture", usage={})
        resp = requests.post(url, headers=headers, json=payload, timeout=self.timeout)
        resp.raise_for_status()
        body = resp.json()
        text = body.get("choices", [{}])[0].get("message", {}).get("content", "")
        usage = body.get("usage", {})
        model_used = body.get("model", self.model)
        return AIResponse(text=text.strip(), model=model_used, usage=usage)


# ── Mock backend (no key, no network) ──

class MockAIClient:
    """Keyword stand-in that satisfies AIClient. Used on the core swap path."""

    def __init__(self, model: str = "mock-keywords"):
        self.model = model

    def complete(self, request: AIRequest) -> AIResponse:
        text = f"{request.prompt}".lower()
        if any(w in text for w in ("charg", "invoice", "refund", "payment", "bill")):
            label = "billing"
        elif any(w in text for w in ("crash", "error", "bug", "500", "stack")):
            label = "technical"
        elif any(w in text for w in ("login", "password", "account", "locked")):
            label = "account"
        else:
            label = "other"
        return AIResponse(text=label, model=self.model, usage={})


print("Shared types, HostedAIClient, and MockAIClient defined.")

Shared types, HostedAIClient, and MockAIClient defined.


## Step 2 — The Test Dataset

A real swap comparison needs **identical inputs** sent to both backends. We'll use a small set of support tickets with known expected categories. This is our **baseline** — if we know the right answer, we can measure accuracy alongside disagreements.

In [5]:
test_rows = [
    {"id": 1, "ticket": "I was charged twice this month",            "expected": "billing"},
    {"id": 2, "ticket": "My login keeps failing with error 403",     "expected": "technical"},
    {"id": 3, "ticket": "How do I change my email address?",         "expected": "account"},
    {"id": 4, "ticket": "The app crashes when I upload a photo",     "expected": "technical"},
    {"id": 5, "ticket": "I need a refund for my last payment",       "expected": "billing"},
    {"id": 6, "ticket": "Can I transfer my subscription to a friend?", "expected": "account"},
]

print(f"Test dataset loaded: {len(test_rows)} rows")

Test dataset loaded: 6 rows


## Step 3 — The Versioned Prompt Config

From **mini-typed-prompt-parser**, we know prompts should be versioned data, not loose strings. We define a single `PromptConfig` and use it for both backends — this guarantees they receive the *exact same instructions*.

In [6]:
@dataclass
class PromptConfig:
    """A versioned, structured prompt configuration."""
    name: str
    version: str
    system: str
    user_template: str
    temperature: float = 0.0
    max_tokens: int = 20

    def render(self, **kwargs) -> str:
        return self.user_template.format(**kwargs)


classifier_prompt = PromptConfig(
    name="ticket-classifier",
    version="2.0",
    system="You are a support ticket classifier. Reply with exactly one word: billing, technical, account, or other.",
    user_template="Classify this support ticket: '{ticket}'\nCategory:",
    temperature=0.0,
    max_tokens=10,
)

print(f"Prompt: {classifier_prompt.name} v{classifier_prompt.version} (max_tokens={classifier_prompt.max_tokens})")

Prompt: ticket-classifier v2.0 (max_tokens=10)


## Step 4 — The Runner: Same Pipeline, Any Backend

This is the core of the swap comparison. The `run_comparison` function takes an `AIClient` (any backend) and runs every test row through it, recording the response and the time taken. The function has **zero vendor imports** — it only knows about the `AIClient` protocol.

In [7]:
VALID_CATEGORIES = {"billing", "technical", "account", "other"}


def normalize_category(raw: str) -> str:
    """Defensively extract a valid category from raw model output."""
    cleaned = raw.strip().lower().rstrip(".!,")
    # Sometimes the model returns extra words; grab the first valid one
    for word in cleaned.split():
        if word in VALID_CATEGORIES:
            return word
    return "other"


def run_comparison(ai_client: AIClient, rows: list, config: PromptConfig) -> list:
    """Run all rows through a backend. Returns a list of result dicts."""
    results = []
    for row in rows:
        request = AIRequest(
            prompt=config.render(ticket=row["ticket"]),
            system=config.system,
            temperature=config.temperature,
            max_tokens=config.max_tokens,
        )

        start = time.time()
        error = None
        try:
            response = ai_client.complete(request)
            raw = response.text
            model = response.model
            predicted = normalize_category(raw)
        except Exception as e:
            # A failing backend is recorded, not fatal
            raw = ""
            model = "N/A"
            predicted = "error"
            error = str(e)
        elapsed = time.time() - start

        results.append({
            "id": row["id"],
            "expected": row["expected"],
            "predicted": predicted,
            "raw_response": raw,
            "model": model,
            "latency_s": elapsed,
            "error": error,
        })
    return results


print("run_comparison() defined.")

run_comparison() defined.


## Step 5 — Run the hosted-or-fixture backend

Send every ticket through `HostedAIClient`. If `OPENAI_API_KEY` is missing, the adapter should return recorded fixtures (see the hosted mini). Do not stop the lab to buy a key.


In [8]:
hosted = HostedAIClient(
    api_key=os.getenv("OPENAI_API_KEY", "sk-fixture"),
    model="gpt-4o-mini",
    timeout=15,
)

hosted_results = run_comparison(hosted, test_rows, classifier_prompt)
print(f"Hosted run done: {len(hosted_results)} rows ({hosted_results[0]['model']})")

Hosted run done: 6 rows (gpt-4o-mini-2024-07-18)


## Step 6 — Run the mock backend

Run the same rows through `MockAIClient` (no network) to compare against hosted-or-fixture.

In [9]:
# The swap: only the constructor changes, run_comparison() stays the same
mock = MockAIClient()

mock_results = run_comparison(mock, test_rows, classifier_prompt)
print(f"Mock run done: {len(mock_results)} rows ({mock.model})")

Mock run done: 6 rows (mock-keywords)


## Step 7 — Record the Disagreements

The whole point of a swap comparison: **where do the two backends disagree**, and what does that tell us about each model's strengths and weaknesses?

In [11]:
# Build the diff record: one entry per row, both backends side by side
diff_records = []
for h, m in zip(hosted_results, mock_results):
    diff_records.append({
        "id": h["id"],
        "expected": h["expected"],
        "hosted_pred": h["predicted"],
        "mock_pred": m["predicted"],
        "agree": h["predicted"] == m["predicted"],
        "hosted_raw": h["raw_response"],
        "mock_raw": m["raw_response"],
    })

agreements = sum(1 for d in diff_records if d["agree"])
disagreements = len(diff_records) - agreements

print(f"Agreements: {agreements}/{len(diff_records)} | Disagreements: {disagreements}")
for d in diff_records:
    if not d["agree"]:
        print(f"  [{d['id']}] expected={d['expected']} hosted={d['hosted_pred']} mock={d['mock_pred']}")

Agreements: 4/6 | Disagreements: 2
  [3] expected=account hosted=account mock=other
  [6] expected=account hosted=account mock=other


In [12]:
# Check a specific disagreement's raw responses
for d in diff_records:
    if not d['agree'] and d['hosted_pred'] != 'error':
        print(f"Ticket: {test_rows[d['id'] - 1]['ticket']}")
        print(f"Hosted raw: {d['hosted_raw']}")
        print(f"Mock raw:   {d['mock_raw']}")
        break

Ticket: How do I change my email address?
Hosted raw: account
Mock raw:   other


## Step 8 — Accuracy & Latency Comparison

Beyond disagreements, let's compare the two backends on **accuracy** (how often each matches the expected label) and **latency** (how fast each responds).

In [13]:
total = len(diff_records)

# Accuracy: how often each backend matches the expected label
hosted_correct = sum(1 for d in diff_records if d["hosted_pred"] == d["expected"])
mock_correct = sum(1 for d in diff_records if d["mock_pred"] == d["expected"])

# Latency: average seconds per row
hosted_avg_latency = sum(r["latency_s"] for r in hosted_results) / total
mock_avg_latency = sum(r["latency_s"] for r in mock_results) / total

md(f"""
| Metric | Hosted ({hosted_results[0]['model']}) | Mock ({mock_results[0]['model']}) |
|--------|--------|------|
| **Accuracy** | {hosted_correct}/{total} ({100*hosted_correct/total:.0f}%) | {mock_correct}/{total} ({100*mock_correct/total:.0f}%) |
| **Avg Latency** | {hosted_avg_latency:.2f}s | {mock_avg_latency:.2f}s |
""")


| Metric | Hosted (gpt-4o-mini-2024-07-18) | Mock (mock-keywords) |
|--------|--------|------|
| **Accuracy** | 6/6 (100%) | 4/6 (67%) |
| **Avg Latency** | 0.94s | 0.00s |


## Step 9 — Save the Comparison Record

Save prompt config + diff to JSON so you can commit it and reproduce comparisons.

In [14]:
comparison_record = {
    "prompt_config": asdict(classifier_prompt),
    "backends": {
        "hosted": {"type": "HostedAIClient", "model": hosted.model},
        "mock": {"type": "MockAIClient", "model": mock.model},
    },
    "results": diff_records,
    "summary": {
        "total_rows": total,
        "agreements": agreements,
        "disagreements": disagreements,
        "hosted_accuracy": hosted_correct / total,
        "mock_accuracy": mock_correct / total,
        "hosted_avg_latency_s": round(hosted_avg_latency, 2),
        "mock_avg_latency_s": round(mock_avg_latency, 2),
    },
}

output_path = "swap_comparison_record.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(comparison_record, f, indent=2, default=str)

print(f"✅ Comparison record saved to {output_path}")

✅ Comparison record saved to swap_comparison_record.json


In [15]:
# Quick peek at the saved file
import json
with open(output_path) as f:
    saved = json.load(f)
print(f"Saved {len(saved['results'])} result rows")
print(f"Prompt: {saved['prompt_config']['name']} v{saved['prompt_config']['version']}")
print(f"Disagreements: {saved['summary']['disagreements']}")

Saved 6 result rows
Prompt: ticket-classifier v2.0
Disagreements: 2


## 🎯 Summary

### What You Built

A recorded swap comparison that runs identical support tickets through a hosted-or-fixture backend and a mock backend, records disagreements, compares accuracy and latency, and saves everything to a JSON file — without changing any business logic between backends.

### Key Takeaways

1. **Interface seam in action** — `run_comparison()` accepted any `AIClient`; swapping hosted for mock changed only the constructor call
2. **Recorded disagreements** — row-by-row comparison shows exactly where backends differ
3. **Versioned prompt + saved record** — the prompt config sits next to the diff, so every comparison is reproducible
4. **Baseline comparison** — expected labels let you judge accuracy and latency, not just agreement
5. **Defensive output parsing** — `normalize_category()` handles extra words and punctuation so the comparison is fair

Tag `v3` after the JSON record exists.